# 01 – Data Exploration

## Turkish Day-Ahead Electricity Market

The **Day-Ahead Market (DAM)** in Turkey is operated by EPIAS (Energy Exchange Istanbul).
Generators and retailers submit bids one day in advance, and the market clears an hourly
price for each hour of the following day.

This notebook explores 6 years (2020–2025) of hourly DAM prices and weather data to
identify temporal patterns, seasonality structures, and the price–temperature relationship.
These patterns directly motivate the feature engineering choices in notebook 02.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/processed/hourly_market.parquet')
df['ts'] = pd.to_datetime(df['ts'])
df['year'] = df['ts'].dt.year
df['month'] = df['ts'].dt.month
df['hour'] = df['ts'].dt.hour
df['day_of_week'] = df['ts'].dt.dayofweek

print(f"Shape: {df.shape}")
print(f"Date range: {df['ts'].min()} → {df['ts'].max()}")
print(f"\nPrice stats (TL/MWh):")
print(df['dam_price_try_mwh'].describe().round(1))


Shape: (52608, 9)
Date range: 2020-01-01 00:00:00 → 2025-12-31 23:00:00

Price stats (TL/MWh):
count    52608.0
mean      1045.8
std        425.9
min        236.1
25%        679.7
50%       1043.7
75%       1407.1
max       1874.4
Name: dam_price_try_mwh, dtype: float64


## 1. Full Price Time Series

In [2]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df['ts'], df['dam_price_try_mwh'], color='#1565C0', linewidth=0.4, alpha=0.7)
monthly_mean = df.resample('ME', on='ts')['dam_price_try_mwh'].mean()
ax.plot(monthly_mean.index, monthly_mean.values, color='#E53935', linewidth=2, label='Monthly avg')
ax.set_title('Turkish DAM Electricity Price — 6 Years (2020–2025)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('DAM Price (TL/MWh)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/01_full_timeseries.png', dpi=150)
plt.show()
print("Saved.")


Saved.


## 2. Hourly Seasonality (Average by Hour of Day)

In [3]:
hourly_avg = df.groupby('hour')['dam_price_try_mwh'].mean()
hourly_std = df.groupby('hour')['dam_price_try_mwh'].std()

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(hourly_avg.index,
                hourly_avg - hourly_std,
                hourly_avg + hourly_std,
                alpha=0.2, color='#1565C0', label='±1 std')
ax.plot(hourly_avg.index, hourly_avg.values, color='#1565C0', linewidth=2, label='Mean')
ax.axvspan(8, 20, alpha=0.08, color='orange', label='Peak hours (08–20)')
ax.set_title('Average DAM Price by Hour of Day', fontsize=13)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Price (TL/MWh)')
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/01_hourly_pattern.png', dpi=150)
plt.show()


## 3. Monthly Seasonality

In [4]:
monthly_box = [df[df['month'] == m]['dam_price_try_mwh'].values for m in range(1, 13)]
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 4))
bp = ax.boxplot(monthly_box, labels=month_labels, patch_artist=True,
                medianprops=dict(color='white', linewidth=2))
colors = plt.cm.coolwarm(np.linspace(0, 1, 12))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('DAM Price Distribution by Month', fontsize=13)
ax.set_ylabel('Price (TL/MWh)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/01_monthly_boxplot.png', dpi=150)
plt.show()


## 4. Weekly Pattern (Avg by Day of Week)

In [5]:
dow_avg = df.groupby('day_of_week')['dam_price_try_mwh'].mean()
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dow_labels, dow_avg.values, color=['#1565C0']*5 + ['#E53935']*2, alpha=0.85)
ax.set_title('Average DAM Price by Day of Week', fontsize=13)
ax.set_ylabel('Avg Price (TL/MWh)')
ax.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, dow_avg.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5, f'{val:.0f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../results/figures/01_dow_pattern.png', dpi=150)
plt.show()
print(f"Weekend discount: {(1 - dow_avg[[5,6]].mean()/dow_avg[:5].mean())*100:.1f}%")


Weekend discount: 9.8%


## 5. Price–Temperature Correlation (Istanbul)

In [6]:
df_corr = df[['dam_price_try_mwh','temp_istanbul']].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Scatter
axes[0].scatter(df_corr['temp_istanbul'], df_corr['dam_price_try_mwh'],
                alpha=0.02, color='#1565C0', s=1)
z = np.polyfit(df_corr['temp_istanbul'], df_corr['dam_price_try_mwh'], 2)
p = np.poly1d(z)
temp_range = np.linspace(df_corr['temp_istanbul'].min(), df_corr['temp_istanbul'].max(), 100)
axes[0].plot(temp_range, p(temp_range), color='#E53935', linewidth=2)
axes[0].set_xlabel('Temperature (°C) — Istanbul')
axes[0].set_ylabel('DAM Price (TL/MWh)')
axes[0].set_title('Temperature vs Price (all hours)')
axes[0].grid(True, alpha=0.3)

# Heatmap: month × hour mean price
pivot = df.pivot_table(values='dam_price_try_mwh', index='month', columns='hour', aggfunc='mean')
im = axes[1].imshow(pivot.values, aspect='auto', cmap='YlOrRd', origin='lower')
axes[1].set_xticks(range(0, 24, 4))
axes[1].set_xticklabels(range(0, 24, 4))
axes[1].set_yticks(range(12))
axes[1].set_yticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_xlabel('Hour of Day')
axes[1].set_title('Average Price: Month × Hour')
plt.colorbar(im, ax=axes[1], label='TL/MWh')
plt.tight_layout()
plt.savefig('../results/figures/01_price_temp_heatmap.png', dpi=150)
plt.show()

corr = df_corr.corr().iloc[0,1]
print(f"Linear correlation (temp, price): {corr:.3f}")


Linear correlation (temp, price): 0.109


## Key Observations

- **Strong hourly pattern**: Morning ramp (08:00) and evening peak (18:00–20:00)
- **Summer premium**: August–September highest prices (cooling demand)
- **Weekend discount**: ~8–10% lower than weekday average
- **Secular price growth**: TL/MWh price has grown 5–6× from 2020 to 2025 (TL inflation)
- **U-shaped temperature effect**: very cold and very hot temperatures both push prices higher